# Mistral 7B — loading, inspecting, comparing

Loading a real 7-billion-parameter LLM, looking inside its layers, and lining it up directly against the ~10M-parameter GPT built from scratch in `lab01_nanogpt`. Same building blocks — token embeddings, attention, feedforward blocks, a language modeling head — just at a completely different scale.

Needs a GPU: 4-bit quantization requires `bitsandbytes` + CUDA. A free Kaggle T4 has more than enough VRAM for this (Mistral 7B in 4-bit needs roughly 4-5GB).

In [ ]:
# Kaggle's base image doesn't ship bitsandbytes by default
!pip install -q -U bitsandbytes accelerate

In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"device: {device}")
if device == 'cpu':
    print("WARNING: no GPU detected. 4-bit loading needs CUDA — this notebook is meant to run on a Kaggle T4.")

## Why 4-bit

Mistral 7B's weights, stored normally (fp16), take about 14GB — more than a free T4's 16GB leaves room for once you account for activations and the KV cache. 4-bit quantization (`nf4`, the format used below) compresses each weight down to roughly half a byte instead of two, shrinking the footprint to about 4-5GB with a small, generally acceptable quality cost. This is exactly how most people run 7B-class models on free or consumer hardware.

In [ ]:
MODEL_ID = "mistralai/Mistral-7B-v0.1"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
)
print(f"Loaded in {time.time() - t0:.1f}s")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Inspecting the layers

`model.config` holds the real architecture numbers straight from the model that was just loaded — no guessing, no hardcoding.

In [ ]:
cfg = model.config

print(f"hidden_size (n_embd)          : {cfg.hidden_size}")
print(f"num_hidden_layers (n_layer)   : {cfg.num_hidden_layers}")
print(f"num_attention_heads (n_head)  : {cfg.num_attention_heads}")
print(f"num_key_value_heads           : {cfg.num_key_value_heads}  (grouped-query attention)")
print(f"intermediate_size (FFN width) : {cfg.intermediate_size}")
print(f"vocab_size                    : {cfg.vocab_size}")
print(f"max_position_embeddings       : {cfg.max_position_embeddings}")
print(f"rope_theta                    : {cfg.rope_theta}")

In [ ]:
# One decoder layer, unfolded — same three ingredients as our Block class, different names
print(model.model.layers[0])

## Same architecture, different choices

Every piece here has a direct equivalent in `lab01_nanogpt`. The differences are the specific engineering choices modern production models make:

| Piece | Our nanoGPT | Mistral 7B |
| --- | --- | --- |
| Normalization | `LayerNorm` | `RMSNorm` (drops the mean-centering step, cheaper to compute) |
| Position encoding | Learned embedding table, added once | RoPE — rotates Q/K inside every attention layer (we built this in lab01!) |
| Attention | Every head has its own K and V | Grouped-Query Attention — several query heads share one K/V head, cutting memory |
| Feedforward | `Linear -> ReLU -> Linear` | SwiGLU — a gated variant with three matrices instead of two |
| Scale | 6 layers, 384 dims, ~10M params | 32 layers, 4096 dims, ~7.3B params |

The RoPE row is worth pausing on: the position encoding we implemented from scratch in lab01 as "an alternative worth comparing" is the position encoding Mistral actually ships with in production.

## Counting parameters

`sum(p.numel() for p in model.parameters())` on a 4-bit-loaded model undercounts — bitsandbytes packs two 4-bit weights into a single byte, so the raw tensor shapes don't reflect the true dense parameter count. Instead, the cell below reconstructs the count from the architecture itself (embedding + per-layer attention + per-layer SwiGLU FFN + LM head), the same kind of calculation used to plan a model's size before training it.

In [ ]:
head_dim = cfg.hidden_size // cfg.num_attention_heads

q_params = cfg.hidden_size * cfg.hidden_size
k_params = cfg.hidden_size * (cfg.num_key_value_heads * head_dim)
v_params = cfg.hidden_size * (cfg.num_key_value_heads * head_dim)
o_params = cfg.hidden_size * cfg.hidden_size
attn_params_per_layer = q_params + k_params + v_params + o_params

# SwiGLU: gate_proj + up_proj (hidden -> intermediate) + down_proj (intermediate -> hidden)
ffn_params_per_layer = 3 * cfg.hidden_size * cfg.intermediate_size

params_per_layer = attn_params_per_layer + ffn_params_per_layer
total_layer_params = cfg.num_hidden_layers * params_per_layer

embedding_params = cfg.vocab_size * cfg.hidden_size
lm_head_params = cfg.vocab_size * cfg.hidden_size

estimated_total = total_layer_params + embedding_params + lm_head_params

print(f"Attention params / layer : {attn_params_per_layer / 1e6:.1f}M")
print(f"FFN params / layer       : {ffn_params_per_layer / 1e6:.1f}M")
print(f"All {cfg.num_hidden_layers} layers            : {total_layer_params / 1e9:.2f}B")
print(f"Embedding + LM head      : {(embedding_params + lm_head_params) / 1e9:.2f}B")
print(f"Estimated total           : {estimated_total / 1e9:.2f}B parameters")
print(f"(Published figure for Mistral-7B-v0.1: ~7.24B — this formula should land close to it)")

## nanoGPT vs Mistral 7B

In [ ]:
nanogpt = {
    "n_layer": 6,
    "n_head": 6,
    "n_embd": 384,
    "vocab_size": 65,
    "context length": 256,
    "position encoding": "learned embedding",
    "normalization": "LayerNorm",
    "feedforward": "Linear-ReLU-Linear",
    "parameters": "10.79M",
}

mistral = {
    "n_layer": cfg.num_hidden_layers,
    "n_head": cfg.num_attention_heads,
    "n_embd": cfg.hidden_size,
    "vocab_size": cfg.vocab_size,
    "context length": cfg.max_position_embeddings,
    "position encoding": "RoPE",
    "normalization": "RMSNorm",
    "feedforward": "SwiGLU",
    "parameters": f"{estimated_total / 1e9:.2f}B",
}

rows = list(nanogpt.keys())
col_w = max(len(r) for r in rows) + 2

print(f"{'Property':<{col_w}} {'nanoGPT (lab01)':<22} {'Mistral 7B'}")
print("-" * (col_w + 22 + 20))
for r in rows:
    print(f"{r:<{col_w}} {str(nanogpt[r]):<22} {mistral[r]}")

ratio = estimated_total / 10.79e6
print(f"\nMistral 7B has ~{ratio:,.0f}x more parameters than our nanoGPT.")

## Temperature — 0.0, 0.8, 1.5

Same prompt, same model, only the temperature changes. `temperature=0.0` means greedy decoding (`do_sample=False`) — always the single most likely next token, fully deterministic. Everything above 0 samples from the softened probability distribution, same mechanism as in lab01's `generate()`, just at Mistral's scale.

In [ ]:
def generate(prompt, temperature, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    gen_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)

    if temperature == 0.0:
        gen_kwargs["do_sample"] = False
    else:
        gen_kwargs["do_sample"] = True
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = 0.95

    with torch.no_grad():
        output_ids = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


prompt = "The most important thing about attention in transformers is"

for temp in [0.0, 0.8, 1.5]:
    print(f"--- temperature = {temp} ---")
    print(generate(prompt, temp))
    print()

### What to look for

- **0.0**: the same output every time you re-run this cell — fully deterministic, always picks the single most likely token.
- **0.8**: coherent but no longer identical on every run — enough variety to feel natural without losing the thread.
- **1.5**: noticeably less coherent, sometimes drifting off-topic or producing less likely word choices — the distribution has been flattened enough that low-probability tokens get picked regularly.

This is the exact same mechanism (`logits / temperature`, then softmax) implemented by hand in lab01 — seeing it on a 7B model confirms the behavior holds at any scale, not just on a 10M-parameter toy model.

## Takeaways

Nothing here is conceptually new — it's the same tokenizer -> embeddings -> attention -> feedforward -> LM head pipeline from `lab01_nanogpt`, at production scale. The real differences are engineering refinements chosen for efficiency at scale: RMSNorm instead of LayerNorm, RoPE instead of a learned position table, grouped-query attention instead of full multi-head, SwiGLU instead of a plain ReLU MLP.

None of that is a black box anymore — it's a bigger version of something already built and understood from scratch.